In [2]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import load_img
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

In [3]:
train_dir = "images/train"
test_dir = "images/test"

In [4]:
def load_data(directory):
    images = []
    labels = []

    for label in os.listdir(directory):
        path = os.path.join(directory, label)

        for img_name in os.listdir(path):
            images.append(os.path.join(path, img_name))
            labels.append(label)

    return pd.DataFrame({"image": images, "label": labels})
train = load_data(train_dir)
test = load_data(test_dir)

print(train.head())

                          image  label
0      images/train\angry\0.jpg  angry
1      images/train\angry\1.jpg  angry
2     images/train\angry\10.jpg  angry
3  images/train\angry\10002.jpg  angry
4  images/train\angry\10016.jpg  angry


In [5]:
def extract_features(image_paths):
    features = []

    for img_path in tqdm(image_paths):
        img = load_img(img_path, color_mode="grayscale", target_size=(48,48))
        img = np.array(img)
        features.append(img)

    return np.array(features).reshape(len(features), 48, 48, 1)

In [6]:
x_train = extract_features(train["image"])
x_test = extract_features(test["image"])

x_train = x_train / 255.0
x_test = x_test / 255.0

100%|██████████| 7066/7066 [00:04<00:00, 1458.52it/s]


In [7]:
le = LabelEncoder()
le.fit(train["label"])

y_train = le.transform(train["label"])
y_test = le.transform(test["label"])

y_train = to_categorical(y_train)
y_test = to_categorical(y_test)

In [8]:
model = Sequential()

model.add(Conv2D(64, (3,3), activation="relu", input_shape=(48,48,1)))
model.add(MaxPooling2D(2,2))
model.add(Dropout(0.2))

model.add(Conv2D(128, (3,3), activation="relu"))
model.add(MaxPooling2D(2,2))
model.add(Dropout(0.3))

model.add(Flatten())

model.add(Dense(128, activation="relu"))
model.add(Dropout(0.3))

model.add(Dense(7, activation="softmax"))

In [9]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [10]:
history = model.fit(
    x_train,
    y_train,
    validation_data=(x_test, y_test),
    epochs=10,
    batch_size=64
)

Epoch 1/10
451/451 [==============================] - 139s 305ms/step - loss: 1.6792 - accuracy: 0.3369 - val_loss: 1.5349 - val_accuracy: 0.4258
Epoch 2/10
451/451 [==============================] - 116s 257ms/step - loss: 1.4866 - accuracy: 0.4284 - val_loss: 1.3759 - val_accuracy: 0.4806
Epoch 3/10
451/451 [==============================] - 115s 255ms/step - loss: 1.3923 - accuracy: 0.4664 - val_loss: 1.3313 - val_accuracy: 0.4967
Epoch 4/10
451/451 [==============================] - 115s 255ms/step - loss: 1.3307 - accuracy: 0.4917 - val_loss: 1.2817 - val_accuracy: 0.5166
Epoch 5/10
451/451 [==============================] - 113s 250ms/step - loss: 1.2799 - accuracy: 0.5092 - val_loss: 1.2720 - val_accuracy: 0.5190
Epoch 6/10
451/451 [==============================] - 113s 251ms/step - loss: 1.2385 - accuracy: 0.5301 - val_loss: 1.2447 - val_accuracy: 0.5284
Epoch 7/10
451/451 [==============================] - 115s 255ms/step - loss: 1.1999 - accuracy: 0.5405 - val_loss: 1.2168 -

In [11]:
history2 = model.fit(
    x_train,
    y_train,
    validation_data=(x_test, y_test),
    epochs=10,
    batch_size=64
)

Epoch 1/10
451/451 [==============================] - 125s 278ms/step - loss: 1.0663 - accuracy: 0.5917 - val_loss: 1.2114 - val_accuracy: 0.5422
Epoch 2/10
451/451 [==============================] - 117s 259ms/step - loss: 1.0394 - accuracy: 0.6081 - val_loss: 1.2117 - val_accuracy: 0.5449
Epoch 3/10
451/451 [==============================] - 117s 258ms/step - loss: 1.0110 - accuracy: 0.6140 - val_loss: 1.1960 - val_accuracy: 0.5529
Epoch 4/10
451/451 [==============================] - 115s 256ms/step - loss: 0.9801 - accuracy: 0.6236 - val_loss: 1.2110 - val_accuracy: 0.5560
Epoch 5/10
451/451 [==============================] - 124s 274ms/step - loss: 0.9559 - accuracy: 0.6354 - val_loss: 1.1890 - val_accuracy: 0.5604
Epoch 6/10
451/451 [==============================] - 120s 266ms/step - loss: 0.9369 - accuracy: 0.6419 - val_loss: 1.2083 - val_accuracy: 0.5617
Epoch 7/10
451/451 [==============================] - 117s 259ms/step - loss: 0.9109 - accuracy: 0.6521 - val_loss: 1.2204 -

In [12]:
model.save("emotion_model.h5")

c:\Users\Faraz\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [13]:
labels = le.classes_

pred = model.predict(x_test[0:1])
print(labels[np.argmax(pred)])

1/1 [==============================] - 1s 1s/step
neutral


In [14]:
from tensorflow.keras.models import load_model

model = load_model("emotion_model.h5")

model.fit(
    x_train,
    y_train,
    validation_data=(x_test, y_test),
    epochs=5,
    batch_size=64
)

Epoch 1/5
451/451 [==============================] - 310s 587ms/step - loss: 0.8430 - accuracy: 0.6778 - val_loss: 1.2445 - val_accuracy: 0.5664
Epoch 2/5
451/451 [==============================] - 122s 269ms/step - loss: 0.8253 - accuracy: 0.6833 - val_loss: 1.2421 - val_accuracy: 0.5626
Epoch 3/5
451/451 [==============================] - 112s 248ms/step - loss: 0.8099 - accuracy: 0.6923 - val_loss: 1.2709 - val_accuracy: 0.5659
Epoch 4/5
451/451 [==============================] - 121s 269ms/step - loss: 0.7768 - accuracy: 0.7047 - val_loss: 1.2673 - val_accuracy: 0.5614
Epoch 5/5
451/451 [==============================] - 132s 293ms/step - loss: 0.7761 - accuracy: 0.7030 - val_loss: 1.2657 - val_accuracy: 0.5634


In [15]:
model.save("emotion_model.h5")